# Train Your Spam Classifier

In this notebook, you will train the machine-learning model that powers your local Spam Classifier app.

`data.csv` → training → `spam_classifier.joblib` → Gradio app

You will load labeled examples, split them fairly, convert text to numbers, train and evaluate a classifier, investigate its behavior, and export it.

## 1. Load the dataset

Models learn from examples. Each row has a message (the input) and a label (the correct answer). This local version reads `data.csv` from the project folder.

In [ ]:
from pathlib import Path

DATA_PATH = Path('data.csv')
assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH.resolve()}'

In [ ]:
import pandas as pd

df = pd.read_csv('data.csv')
df.head()

## 2. Explore the data

Before training, check how many examples there are, which labels exist, whether the classes are balanced, and whether the messages look realistic.

In [ ]:
print(f'Total examples: {len(df)}')
print('\nExamples per label:')
print(df['label'].value_counts())

df.sample(10)

## 3. Split training and testing data

We hold out 20% of the examples for testing. Otherwise, we could only tell whether the model remembers its training data.


In [ ]:
from sklearn.model_selection import train_test_split

X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Training examples: {len(X_train)}')
print(f'Testing examples: {len(X_test)}')

## 4. Turn text into numbers

A traditional classifier cannot directly use words such as `free` or `meeting`; it needs numerical features. TF-IDF represents text using the words it contains and how informative they are across the dataset.

Our pipeline combines TF-IDF with Logistic Regression, which learns patterns associated with each label. We keep stop words because short-message words such as `your`, `account`, and `free` can be useful spam signals.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words=None, ngram_range=(1, 1))),
    ('classifier', LogisticRegression(max_iter=3000)),
])

param_grid = {'classifier__C': [0.3, 1.0, 3.0, 10.0, 30.0]}
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipeline

## 5. Train the model

This is the moment the general algorithm learns parameters from your examples. The regularization value C is selected using only the training set, so the held-out test set remains an honest final check.

In [ ]:
search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv_strategy,
    n_jobs=-1,
    refit=True,
)
search.fit(X_train, y_train)
pipeline = search.best_estimator_

print(f'Best C selected on training data: {search.best_params_["classifier__C"]}')
print(f'Best training CV macro-F1: {search.best_score_:.3f}')

## 6. Evaluate the model

Accuracy tells us how often the model got the held-out dataset correct. It is not the same as the confidence of any one prediction.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

predictions = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
macro_f1 = f1_score(y_test, predictions, average='macro')
spam_precision = precision_score(y_test, predictions, pos_label='spam')
spam_recall = recall_score(y_test, predictions, pos_label='spam')

print(f'Test accuracy: {accuracy:.1%}')
print(f'Test macro-F1: {macro_f1:.3f}')
print(f'Spam precision: {spam_precision:.3f}')
print(f'Spam recall: {spam_recall:.3f}')
print('\nConfusion matrix [not_spam, spam]:')
print(confusion_matrix(y_test, predictions, labels=['not_spam', 'spam']))
print('\nDetailed report:')
print(classification_report(y_test, predictions, digits=3))

## 6.1. Check model stability

A single train/test split is useful for comparison but can be noisy with a small dataset. This nested evaluation selects the regularization parameter inside each training fold, then evaluates on untouched outer folds.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate

nested_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words=None, ngram_range=(1, 1))),
    ('classifier', LogisticRegression(max_iter=3000)),
])
nested_search = GridSearchCV(
    nested_pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=cv_strategy,
    n_jobs=-1,
)
outer_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=123)
nested_results = cross_validate(
    nested_search,
    X,
    y,
    cv=outer_cv,
    scoring=['accuracy', 'f1_macro'],
    n_jobs=1,
)

print(f'Nested CV accuracy: {nested_results["test_accuracy"].mean():.3f} +/- {nested_results["test_accuracy"].std():.3f}')
print(f'Nested CV macro-F1: {nested_results["test_f1_macro"].mean():.3f} +/- {nested_results["test_f1_macro"].std():.3f}')

## 6.2. Inspect mistakes

False positives are normal messages incorrectly flagged as spam. False negatives are spam messages that slipped through. These examples show where a larger or better-trained dataset would help.

In [ ]:
error_table = pd.DataFrame({
    'message': X_test,
    'actual': y_test,
    'predicted': predictions,
}).reset_index(drop=True)
errors = error_table[error_table['actual'] != error_table['predicted']]
print(errors.to_string(index=False))

## 7. Classify your own messages

The probability distribution shows how strongly the model favors each label. A 51% / 49% prediction is much less decisive than 99% / 1%.

In [ ]:
def classify(text):
    prediction = pipeline.predict([text])[0]
    probabilities = pipeline.predict_proba([text])[0]
    return {
        'prediction': prediction,
        'probabilities': {
            label: round(float(probability), 3)
            for label, probability in zip(pipeline.classes_, probabilities)
        },
    }

classify("Congratulations! You've won a FREE vacation!")

## 8. Challenge: try to break the model

Find a normal message labeled spam, a spam message labeled normal, a near-50/50 message, and two nearly identical messages with different confidence. Try these starting points:

- `Congratulations on winning the hackathon!`
- `Free pizza at the CS+AI meeting tonight.`
- `Click this link to download the lecture notes.`
- `URGENT: please send me the homework.`
- `We've been trying to reach you regarding your account.`

The model is learning word patterns from a small dataset, not reasoning about spam as a person does.

## 9. Inspect what the model learned

TF-IDF plus logistic regression is interpretable. Assuming class order is `['not_spam', 'spam']`, positive coefficients are more associated with spam.

In [ ]:
import numpy as np

vectorizer = pipeline.named_steps['tfidf']
classifier = pipeline.named_steps['classifier']
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = classifier.coef_[0]

print('Class order:', pipeline.classes_)

top_spam_indices = np.argsort(coefficients)[-10:][::-1]
top_not_spam_indices = np.argsort(coefficients)[:10]

print('\nWords most associated with SPAM:')
for index in top_spam_indices:
    print(feature_names[index], round(coefficients[index], 3))

print('\nWords most associated with NOT SPAM:')
for index in top_not_spam_indices:
    print(feature_names[index], round(coefficients[index], 3))

## 10. Refit on all data and export the local model

We use the held-out test results for evaluation, then refit the selected pipeline on all labeled examples before exporting. This gives the local app access to every training example while keeping the reported test metrics honest.

In [ ]:
from sklearn.base import clone
import joblib

final_pipeline = clone(pipeline)
final_pipeline.fit(X, y)
pipeline = final_pipeline

MODEL_PATH = Path('model') / 'spam_classifier.joblib'
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(pipeline, MODEL_PATH)
print(f'Saved full-data model as {MODEL_PATH.resolve()}')

In [ ]:
assert MODEL_PATH.exists()
print(f'Model size: {MODEL_PATH.stat().st_size} bytes')

## Return to the app

The trained artifact is already saved at `model/spam_classifier.joblib`. Run `python app.py` from the project folder; the Gradio interface will load the full-data model and display a prediction with probabilities.

Finally, manually compare the same messages with ChatGPT, Gemini, Claude, or another chatbot. Do this outside the Gradio app. Discuss where the systems disagree and whether the more capable model is always the best engineering choice.